# One preregistered Group DRO control

This notebook trains exactly one additional model: the same Balanced low+haze data and NAFNet protocol, with epoch-level Group DRO weights over the two formation orders. The checkpoint is selected by worst-order validation PSNR.

The notebook first reuses `/kaggle/working/order_controls_low_haze_results.zip` from the still-active preceding session. Only if that file is absent does it look for an attached Kaggle Dataset. The old models are not retrained. Expected runtime is roughly 65–80 minutes on 2xT4.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/HoangKhanhTung0111/CoT-restoration.git"
REPO_DIR = Path("/kaggle/working/CoT-restoration")
if REPO_DIR.is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Commit:", COMMIT)

In [ ]:
CDD11_ROOT = Path("/kaggle/input/datasets/mintesnotfikir/cdd-11-30")
PRETRAINED_ROOT = Path("/kaggle/input/datasets/hoangkhanhtung/nafnetmodel")
CONFIG = Path("configs/order_group_dro_low_haze.json")
EXPERIMENTS_ROOT = Path("/kaggle/working/experiments_order_group_dro")
EVALUATION_DIR = Path("/kaggle/working/order_group_dro_evaluation")
working_reference = Path("/kaggle/working/order_controls_low_haze_results.zip")
if working_reference.is_file():
    reference_candidates = [working_reference]
    reference_source = "existing /kaggle/working artifact"
else:
    reference_candidates = sorted(Path("/kaggle/input").rglob("order_controls_low_haze_results.zip"))
    reference_source = "attached Kaggle Dataset fallback"

assert CDD11_ROOT.is_dir(), f"Missing CDD-11 input: {CDD11_ROOT}"
assert PRETRAINED_ROOT.is_dir(), f"Missing pretrained input: {PRETRAINED_ROOT}"
assert CONFIG.is_file(), f"Missing config: {CONFIG}"
assert len(reference_candidates) == 1, (
    "Could not find exactly one prior result ZIP in /kaggle/working or /kaggle/input; "
    f"found {reference_candidates}"
)
REFERENCE_ZIP = reference_candidates[0]
gpu_names = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip().splitlines()
assert len(gpu_names) == 2 and all("T4" in name for name in gpu_names), f"Select Kaggle 2xT4; found {gpu_names}"
print("Reference:", REFERENCE_ZIP, "from", reference_source)
print("GPUs:", gpu_names)

In [ ]:
subprocess.run([
    "python", "-m", "hybrid_cot_nafnet.audit_kaggle",
    "--data-root", str(CDD11_ROOT),
    "--pretrained-root", str(PRETRAINED_ROOT),
    "--output", "/kaggle/working/group_dro_input_audit.json",
], check=True)
subprocess.run([
    "python", "-m", "hybrid_cot_nafnet.audit_order_control_data",
    "--data-root", str(CDD11_ROOT),
    "--output", "/kaggle/working/group_dro_data_gate.json",
], check=True)
subprocess.run([
    "python", "-m", "hybrid_cot_nafnet.run_ablation",
    "--config", str(CONFIG),
    "--data-root", str(CDD11_ROOT),
    "--experiments-root", str(EXPERIMENTS_ROOT),
    "--nproc-per-node", "2",
    "--dry-run",
], check=True)

In [ ]:
subprocess.run([
    "python", "-m", "hybrid_cot_nafnet.run_ablation",
    "--config", str(CONFIG),
    "--data-root", str(CDD11_ROOT),
    "--experiments-root", str(EXPERIMENTS_ROOT),
    "--nproc-per-node", "2",
], check=True)
GROUP_RUN = EXPERIMENTS_ROOT / "order_group_dro_ab_seed42_20ep"
GROUP_CHECKPOINT = GROUP_RUN / "best.pt"
assert GROUP_CHECKPOINT.is_file(), f"Missing checkpoint: {GROUP_CHECKPOINT}"

In [ ]:
subprocess.run([
    "python", "-m", "hybrid_cot_nafnet.evaluate_group_dro_control",
    "--checkpoint", str(GROUP_CHECKPOINT),
    "--reference-zip", str(REFERENCE_ZIP),
    "--data-root", str(CDD11_ROOT),
    "--output-dir", str(EVALUATION_DIR),
    "--realizations", "3",
    "--bootstrap-samples", "5000",
], check=True)

In [ ]:
import pandas as pd
from IPython.display import display

decision = json.loads((EVALUATION_DIR / "decision.json").read_text())
contrasts = json.loads((EVALUATION_DIR / "contrasts.json").read_text())
print(json.dumps(decision, indent=2))
print(json.dumps(contrasts, indent=2))
display(pd.read_csv(EVALUATION_DIR / "per_order_all.csv"))
display(pd.read_csv(GROUP_RUN / "train_log.csv")[["epoch", "val_order_a_psnr", "val_order_b_psnr", "val_worst_order_psnr", "next_group_weight_a", "next_group_weight_b"]])

In [ ]:
import zipfile
from IPython.display import FileLink

required = ["decision.json", "contrasts.json", "per_order_all.csv", "per_sample_all.csv", "protocol.json"]
for name in required:
    assert (EVALUATION_DIR / name).is_file(), f"Missing evaluation artifact: {name}"
output_zip = Path("/kaggle/working/order_group_dro_results.zip")
with zipfile.ZipFile(output_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(EVALUATION_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, Path("evaluation") / path.relative_to(EVALUATION_DIR))
    archive.write(CONFIG, Path("protocol") / CONFIG.name)
    archive.write(Path("docs/degradation_order_control_plan.md"), Path("protocol/degradation_order_control_plan.md"))
    archive.write(Path("/kaggle/working/group_dro_input_audit.json"), Path("protocol/group_dro_input_audit.json"))
    archive.write(Path("/kaggle/working/group_dro_data_gate.json"), Path("protocol/group_dro_data_gate.json"))
    for name in ("run_summary.json", "run_config.json", "dataset_manifest.json", "runtime_resolution.json", "pretrained_report.json", "git_info.json", "train_log.csv", "memory_log.csv"):
        path = GROUP_RUN / name
        if path.is_file():
            archive.write(path, Path("training") / name)
print("Download this ZIP and place it under results/:")
display(FileLink(str(output_zip)))